In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier

# Connect to BigQuery
client = bigquery.Client.from_service_account_json("C:/Users/NMTRAN/.dbt/keyfile.json")
dataset = "USER_ACCQUISITION"

# Load user-level and spend data
query_user = f"""
SELECT *
FROM `{client.project}.{dataset}.user_data`
"""

query_spend = f"""
SELECT
    date,
    platform,
    campaign_name,
    network_name,
    spend
FROM `{client.project}.{dataset}.spend`
"""

user_df = client.query(query_user).to_dataframe()
spend_df = client.query(query_spend).to_dataframe()

C:\Users\NMTRAN\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# Handle iOS Opt-out users using cv_bucket
cv_df = client.query(f"""
SELECT cv_bucket, (rev_min + rev_max)/2 as revenue_est
FROM `{client.project}.{dataset}.dim_cv_bucket`
""").to_dataframe()

In [ ]:
# Assign revenue_est for opt-out users
user_df['revenue_est'] = user_df['revenue_usd']
mask_optout = (user_df.platform == 'ios') & (user_df.tracking_status == 'Opt out')
user_df.loc[mask_optout, 'revenue_est'] = user_df.loc[mask_optout, 'cv_bucket'].map(
    cv_df.set_index('cv_bucket')['revenue_est']
)

In [ ]:
# Fill missing revenue_est with 0
user_df['revenue_est'] = user_df['revenue_est'].fillna(0)

In [ ]:
# Aggregate metrics per campaign
campaign_metrics = user_df.groupby(['platform','campaign_name','network_name']).agg(
    total_revenue = ('revenue_est','sum'),
    total_users = ('adid','nunique')
).reset_index()

# Merge total spend
spend_sum = spend_df.groupby(['platform','campaign_name','network_name'])['spend'].sum().reset_index()
campaign_metrics = campaign_metrics.merge(spend_sum, on=['platform','campaign_name','network_name'], how='left')
campaign_metrics.rename(columns={'spend':'total_spend'}, inplace=True)

# Calculate additional KPIs
campaign_metrics['conversion_rate'] = 100  # as all users considered for revenue
campaign_metrics['roas'] = campaign_metrics['total_revenue'] / campaign_metrics['total_spend']
campaign_metrics['arpu'] = campaign_metrics['total_revenue'] / campaign_metrics['total_users']

In [ ]:
# Forecast revenue per campaign (Linear Regression)
# Create daily revenue per campaign & platform
daily_revenue = user_df.groupby(['platform','campaign_name','install_date'])['revenue_est'].sum().reset_index()

forecast_results = []

for platform in campaign_metrics['platform'].unique():
    for c in campaign_metrics[campaign_metrics['platform']==platform]['campaign_name'].unique():
        df_daily = daily_revenue[(daily_revenue['platform']==platform) & (daily_revenue['campaign_name']==c)]
        
        if len(df_daily) > 1:
            df_daily['install_date'] = pd.to_datetime(df_daily['install_date'])
            X = df_daily['install_date'].map(pd.Timestamp.toordinal).values.reshape(-1,1)
            y = df_daily['revenue_est'].values
            model = LinearRegression()
            model.fit(X, y)
            forecast = model.predict([[X[-1][0]+1]])[0]
        else:
            forecast = df_daily['revenue_est'].iloc[0]
            
        forecast_results.append([platform, c, forecast])

forecast_df = pd.DataFrame(forecast_results, columns=['platform','campaign_name','forecasted_revenue'])
campaign_metrics = campaign_metrics.merge(forecast_df, on=['platform','campaign_name'], how='left')

In [ ]:
# High ROAS probability (simple Random Forest)
# Define target: high ROAS if ROAS > 0.3. To be adjusted based on business needs. Ideally the tier is >1 
campaign_metrics['high_roas'] = (campaign_metrics['roas']>0.3).astype(int)

# Features: total_spend, total_users, arpu
X = campaign_metrics[['total_spend','total_users','arpu']]
y = campaign_metrics['high_roas']

rf_model = RandomForestClassifier(n_estimators=50, random_state=1)
rf_model.fit(X, y)
campaign_metrics['high_roas_prob'] = rf_model.predict_proba(X)[:,1]

C:\Users\NMTRAN\AppData\Local\Temp\ipykernel_19692\1776121071.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily['install_date'] = pd.to_datetime(df_daily['install_date'])
C:\Users\NMTRAN\AppData\Local\Temp\ipykernel_19692\1776121071.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily['install_date'] = pd.to_datetime(df_daily['install_date'])
C:\Users\NMTRAN\AppData\Local\Temp\ipykernel_19692\1776121071.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

In [ ]:
# Export to CSV for dbt seed
campaign_metrics.to_csv(r"C:\Users\NMTRAN\voodoo_case\seeds\campaign_analysis.csv", index=False)
print("Campaign Analysis CSV ready for dbt seed!")
print(campaign_metrics.head())

Campaign Analysis CSV ready for dbt seed!
  platform campaign_name network_name  total_revenue  total_users  \
0  android     campaignA    Network A  148442.629457        55923   
1  android     campaignB    Network A   24995.289201         4070   
2  android     campaignC    Network A   55684.052456        12015   
3      ios     campaignB    Network A  200689.286634       166384   
4      ios     campaignC    Network A   54744.411322        50529   

   total_spend  conversion_rate      roas      arpu  forecasted_revenue  \
0  441736.5056              100  0.336043  2.654411         2755.591174   
1   52589.5928              100  0.475290  6.141349          980.043714   
2  269001.7785              100  0.207003  4.634545         2377.520606   
3  313258.8126              100  0.640650  1.206181         6002.831776   
4  224959.8117              100  0.243352  1.083426         2427.602282   

   high_roas  high_roas_prob  
0          1            0.94  
1          1            0.74  